In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, interactive_output
from IPython.display import display

# Configuration: Fixed maximum time axis length
N_max = 128
n_full = np.arange(N_max)

# 1. High-resolution dense reference signal (for proper spectrum calculation)
f_start, f_end = 0.05, 0.35
t_normalized = np.linspace(0, 1, N_max)
base_signal = np.exp(-3 * t_normalized) * np.sin(2 * np.pi * (f_start + 0.5 * (f_end - f_start) * t_normalized) * n_full)

def plot_rectangular_fixed_time(N):
    plt.close('all')
    
    # --- A) VISUAL PART: Samples spread across the entire interval (densify/thine) ---
    display_indices = np.linspace(0, N_max - 1, N, dtype=int)
    x_display = base_signal[display_indices]
    
    # --- B) COMPUTATIONAL PART: Strict truncation of N consecutive samples for proper spectral spreading ---
    x_finite = base_signal[:N]
    
    # High-resolution FFT
    n_fft = 2048
    freqs = np.fft.fftshift(np.fft.fftfreq(n_fft))
    
    # Finite signal spectrum (with rectangular window of size N)
    X_padded = np.zeros(n_fft, dtype=complex)
    X_padded[:N] = x_finite
    X_spec = np.fft.fftshift(np.fft.fft(X_padded, n_fft))
    
    # Ideal reference spectrum (without truncation, N_max)
    X_ideal_padded = np.zeros(n_fft, dtype=complex)
    X_ideal_padded[:N_max] = base_signal
    X_ideal_spec = np.fft.fftshift(np.fft.fft(X_ideal_padded, n_fft))

    # Plotting layout
    fig = plt.figure(figsize=(16, 9.33))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], hspace=0.4, wspace=0.2)
    
    # Time Domain: Visual distribution along the full length (samples densify/thine)
    ax_t = fig.add_subplot(gs[0, :])
    ax_t.plot(n_full, base_signal, 'k:', alpha=0.4, label='Full Continuous Signal')
    ax_t.stem(display_indices, x_display, linefmt='#1f77b4', markerfmt='o', basefmt='k-')
    ax_t.set_title(f"Time Domain: Visual Spreading (N = {N} samples - Variable Sampling Density)", fontsize=10, fontweight='bold')
    ax_t.set_xlim(-1, N_max)
    ax_t.legend(fontsize=8)
    ax_t.grid(True, linestyle='--')
    
    # Window Spectrum (Sinc of the rectangular window)
    ax_wm = fig.add_subplot(gs[1, 0])
    w_rect = np.ones(N)
    W_padded = np.zeros(n_fft)
    W_padded[:N] = w_rect
    W_spec = np.fft.fftshift(np.fft.fft(W_padded, n_fft))
    W_mag_db = 20 * np.log10(np.abs(W_spec) / np.max(np.abs(W_spec)) + 1e-6)
    ax_wm.plot(freqs, W_mag_db, 'orange')
    ax_wm.set_title(f"Rectangular Window Spectrum (N={N}): Side Lobes & Leakage", fontsize=9)
    ax_wm.set_ylim(-50, 5)
    ax_wm.grid(True, linestyle='--')
    
    ax_wp = fig.add_subplot(gs[2, 0])
    ax_wp.plot(freqs, np.unwrap(np.angle(W_spec)), 'purple')
    ax_wp.set_title("Rectangular Window Phase (Unwrapped)", fontsize=9)
    ax_wp.grid(True, linestyle='--')
    
    # Continuous Spectrum Comparison (Smearing vs Leakage based on proper truncation)
    ax_sm = fig.add_subplot(gs[1, 1])
    
    ref_magnitude = np.abs(X_ideal_spec) / (N_max/N)
    win_magnitude = np.abs(X_spec)
    
    ax_sm.fill_between(freqs, ref_magnitude, color='blue', alpha=0.2, label='Reference (Ideal N=128)')
    ax_sm.fill_between(freqs, win_magnitude, color='red', alpha=0.25, label=f'Rectangular Window N={N}')
    
    ax_sm.plot(freqs, ref_magnitude, 'b--', alpha=0.7, lw=1)
    ax_sm.plot(freqs, win_magnitude, 'r-', lw=1.2)
    
    ax_sm.set_title("Continuous Spectrum: Smearing vs Leakage Trade-off", fontsize=9, fontweight='bold')
    ax_sm.set_xlim(-0.5, 0.5)
    
    ax_sm.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.)
    ax_sm.grid(True, linestyle='--')
    
    # Signal Phase Spectrum
    ax_sp = fig.add_subplot(gs[2, 1])
    ax_sp.plot(freqs, np.unwrap(np.angle(X_spec)), 'g-', label='Windowed Phase')
    ax_sp.set_title("Signal Phase Spectrum (Unwrapped)", fontsize=9, fontweight='bold')
    ax_sp.set_xlim(-0.5, 0.5)
    ax_sp.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.)
    ax_sp.grid(True, linestyle='--')

    # Proper layout management without warnings
    fig.subplots_adjust(right=0.82, top=0.92, bottom=0.08, hspace=0.4, wspace=0.2)
    plt.show()

# Widget for changing N with default value 128
N_slider = IntSlider(value=128, min=16, max=128, step=4, description='Window N:')
display(VBox([N_slider, interactive_output(plot_rectangular_fixed_time, {'N': N_slider})]))